# Scrape Google Trends for Your Own Topic - Lane A (the real experience: you prompt, the agent builds)

**SISMID 2026 - Day 1, 3:30 session.** You drive a coding agent (Codex, Claude Code, or
Antigravity CLI) to pull Google Trends for **your own** disease and place, and keep the
judgment. For each step: read the goal, paste the **prompt**, run the code, apply the check.

> Each prompt produces roughly the matching cell in the **Lane B** notebook. Not set up
> with an agent yet? Use Lane B. Default example topic: flu in the US.


## Step 0: set up a robust fetch, and pick your topic

> *Using pytrends, write `gt_fetch(kw_list, timeframe, geo)` that returns a tidy DataFrame*
> *(date + one column per entry, underscored names) and accepts literal phrases, additive*
> *"a + b" queries, or topic mids. Build TrendReq with retries and a small random pause;*
> *on a 429 wait ~10s and retry a few times; return None (not raise) if it still fails.*
> *Also write `topic_mid(phrase)` that resolves a phrase to a Google Trends topic entity*
> *via pytrends suggestions, and a loader for the cached example CSV in data/. Then set*
> *two variables I will edit: MY_TERMS (my disease's query phrases) and MY_GEO.*


In [ ]:
from pathlib import Path
import random
import time

import pandas as pd
from pytrends.request import TrendReq

# ===== EDIT THESE TWO LINES =====
MY_TERMS = ['dengue', 'fever', 'sintomas de dengue']  # Up to five phrases, queries, or topic mids
MY_GEO = 'BR-RJ'                                      # Brazil, Rio de Janeiro
# ==================================

RECENT_TF = '2024-01-01 2024-12-31'  # Calendar year 2024
CACHE_PATH = Path('../data/google_trends_flu_us_cached.csv')


def _column_name(value):
    """Make a predictable, notebook-friendly name from a Trends query."""
    return '_'.join(str(value).strip().split())


def _is_rate_limited(error):
    message = str(error).lower()
    return '429' in message or 'too many requests' in message


def gt_fetch(kw_list, timeframe, geo, tries=4):
    """Fetch Google Trends interest as date plus one underscored column per query.

    Entries may be literal phrases, additive queries such as ``'flu + gripe'``,
    or Google Trends topic mids such as ``'/m/0cycc'``.  Returns ``None`` after
    repeated failures so a notebook can fall back to its cached example.
    """
    if not 1 <= len(kw_list) <= 5:
        print('Google Trends accepts between 1 and 5 query entries.')
        return None

    time.sleep(random.uniform(0.25, 1.25))  # avoid a room-wide simultaneous request
    for attempt in range(tries):
        try:
            pt = TrendReq(hl='en-US', tz=360, retries=2, backoff_factor=0.5)
            pt.build_payload(kw_list=list(kw_list), timeframe=timeframe, geo=geo)
            frame = pt.interest_over_time()
            if frame.empty:
                raise RuntimeError('Google Trends returned an empty result')

            frame = frame.drop(columns=['isPartial'], errors='ignore').reset_index()
            return frame.rename(columns={column: _column_name(column)
                                         for column in frame.columns})
        except Exception as error:
            if _is_rate_limited(error) and attempt < tries - 1:
                print(f'Google returned 429; waiting 10s before retry {attempt + 2}/{tries}...')
                time.sleep(10)
                continue
            print(f'Google Trends pull failed ({type(error).__name__}): {error}')
            return None
    return None


def topic_mid(phrase):
    """Return ``(mid, title, type)`` for the first Trends topic suggestion."""
    try:
        suggestions = TrendReq(hl='en-US', tz=360).suggestions(phrase)
    except Exception as error:
        print(f'Topic lookup failed ({type(error).__name__}): {error}')
        return phrase, phrase, 'raw term'

    for suggestion in suggestions:
        if suggestion.get('mid'):
            print(f"{phrase!r} -> {suggestion['title']} ({suggestion.get('type', 'topic')}), {suggestion['mid']}")
            return suggestion['mid'], suggestion['title'], suggestion.get('type', 'topic')
    print(f'No Google Trends topic found for {phrase!r}.')
    return phrase, phrase, 'raw term'


def load_cache(path=CACHE_PATH):
    """Load the included flu/US example when a live request is unavailable."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Cached example not found: {path}')
    return pd.read_csv(path, parse_dates=['date'])


## Step 1: pull your terms, live

> *Use gt_fetch to pull MY_TERMS for MY_GEO over the past 5 years (today 5-y). If it*
> *returns None, fall back to the cached example. Plot the series and print the last date.*

**Your check:** does the last point land on the current week, and do peaks fall in a
plausible season for your disease and place?


In [ ]:
import matplotlib.pyplot as plt

LIVE_TIMEFRAME = 'today 5-y'
series = gt_fetch(MY_TERMS, timeframe=LIVE_TIMEFRAME, geo=MY_GEO)
used_cache = series is None
if used_cache:
    series = load_cache()
    print('Live pull was unavailable; plotted the cached flu/US example instead.')

columns = [column for column in series.columns if column != 'date']
print(f'rows: {len(series)} | last data point: {series["date"].max().date()}')

fig, ax = plt.subplots(figsize=(10, 4))
for column in columns:
    ax.plot(series['date'], series[column], label=column)
label = 'cached flu/US example' if used_cache else f'{MY_GEO}, {LIVE_TIMEFRAME}'
ax.set(title=f'Google Trends terms ({label})', xlabel='Date', ylabel='Search interest (0-100)')
ax.legend()
fig.tight_layout()
plt.show()


## Step 2: term vs topic

> *Compare three ways to get the signal: (1) the individual terms, (2) one combined*
> *additive query joining the phrases with ' + ' (a topic-like OR), and (3) the real*
> *Google Trends topic entity, resolved with topic_mid and pulled by its mid. Report the*
> *percent of zeros for each, and overlay them on one plot. Which has the fewest zeros and*
> *the cleanest seasonal signal?*

**Why it matters:** a raw term misses synonyms, spellings, and languages; a topic folds
them in, with more volume and fewer zeros.


In [ ]:
COMPARISON_TF = 'today 5-y'  # Set to RECENT_TF to repeat the 2024 comparison
live = gt_fetch(MY_TERMS, COMPARISON_TF, MY_GEO)
combo_query = ' + '.join(MY_TERMS)
combo = gt_fetch([combo_query], COMPARISON_TF, MY_GEO)
mid, title, kind = topic_mid(MY_TERMS[0])
print(f"{MY_TERMS[0]!r} resolves to -> {title} ({kind}), mid={mid}")
topic = gt_fetch([mid], COMPARISON_TF, MY_GEO) if mid != MY_TERMS[0] else None

if live is not None:
    tcols = [column for column in live.columns if column != 'date']
    pct_zero = (live[tcols] == 0).mean().mul(100).round(1)
    print('\n% zeros, individual terms:')
    print(pct_zero)
    if combo is not None:
        ccol = [column for column in combo.columns if column != 'date'][0]
        print(f'% zeros, combined \"+\" query: {float((combo[ccol] == 0).mean() * 100):.1f}')
    if topic is not None:
        pcol = [column for column in topic.columns if column != 'date'][0]
        print(f'% zeros, topic entity {title}: {float((topic[pcol] == 0).mean() * 100):.1f}')

    fig, ax = plt.subplots(figsize=(10, 4))
    for column in tcols:
        ax.plot(live['date'], live[column], alpha=.5, label=f'term: {column}')
    if combo is not None:
        ax.plot(combo['date'], combo[ccol], 'k', lw=2, label='combined \"+\"')
    if topic is not None:
        ax.plot(topic['date'], topic[pcol], 'r', lw=2, label=f'topic: {title}')
    ax.set(title='Terms vs combined phrases vs topic entity', xlabel='Date', ylabel='Search interest (0-100)')
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print('Live pull blocked; skip the comparison (the cached terms above still work).')


## Step 3: expose the instability

> *Pull the first term twice with the same window and geo, merge on date, and report*
> *whether the two pulls are identical and their mean absolute difference.*


In [ ]:
a = gt_fetch(MY_TERMS[:1], timeframe=RECENT_TF, geo=MY_GEO)
b = gt_fetch(MY_TERMS[:1], timeframe=RECENT_TF, geo=MY_GEO)
if a is not None and b is not None:
    merged = a.merge(b, on='date', suffixes=('_1', '_2'))
    column = [name for name in a.columns if name != 'date'][0]
    difference = (merged[column + '_1'] - merged[column + '_2']).abs()
    print('identical pulls?', bool((difference == 0).all()), '| mean absolute difference:', round(difference.mean(), 2))
else:
    print('Could not compare live (rate-limited). Repeat pulls can differ.')


## Step 4: sanity-check and save

> *Report the geo, date range, row count, missing values per column, and which terms vary.*
> *Then save the tidy table to my_topic_search.csv and confirm it was written.*


In [ ]:
cols = [column for column in series.columns if column != 'date']
print('geo         :', MY_GEO)
print('date range  :', series['date'].min().date(), 'to', series['date'].max().date())
print('n rows      :', len(series))
print('missing per column:')
print(series[cols].isna().sum())
print('which terms actually move (standard deviation)?')
print(series[cols].std().round(2))

out = 'my_topic_search.csv'
series.to_csv(out, index=False)
print('saved', out, 'with', len(series), 'rows and columns', list(series.columns))


## Reflection

- You described the outcome; the agent wrote the pytrends plumbing, retries and all.
- You saw the **term vs topic** difference first-hand, and know when to prefer a combined
  or topic signal.
- **If the whole room keeps getting blocked:** route through a proxy or VPN (see the
  slides and `proxy-setup.md`).
- **Day 2:** the same loop for Wikipedia, wastewater, mobility, weather, and news.
